In [1]:
### 指定輸入參數
maximum_feed_rate = 48000 #最大進給速率, mm/min
motor_max_speed = 4000 #馬達最高轉速，預設3000 rpm
acceleration = "" #加速度
reduction_ratio = 1 #減速比
load = 775 #負載
cutting_force = 343 #切削力
length = 924 # 螺桿長度，兩端軸承間距
preload_rate = 0.05 #預壓率
axis = ["x", "y", "z"]
gravity_axis_YN = True #判斷重力軸
guide = maximum_feed_rate / (motor_max_speed * reduction_ratio)
N = maximum_feed_rate / guide #螺桿最高轉速

In [2]:
from math import pi

# 導程 & 最大轉速，最大進給速率 = 導程 * 最大轉速 * 減速比
guide = maximum_feed_rate / (motor_max_speed * reduction_ratio) #導程

def Diameter_calculation():
    Nm = (maximum_feed_rate / guide) * 0.5 #臨界轉速

    #由導螺桿臨界轉速估算導螺桿桿徑
    f = [9.7, 15.1, 21.9, 3.4] #[支-支, 固-支, 固-固, 固-自]
    dr_n = round((Nm * length**2 / f[2]) * 1e-7, 0) # dr = (n * (length**2) / f) * (10**-7)

    #由挫曲負荷估算導螺桿桿徑
    if gravity_axis_YN:
        p = (load + cutting_force) * 2
    else:
       cof = 0.008 #摩擦力係數
       ff = load * cof 
       p = (cutting_force + ff) * 2
    E = 21000 #kgf/mm2
    n = [4.0, 2.0, 0.25] #[固-支, 固-固, 固-自]
    dr_p = round((p * 64 * (length**2) / (n[0] * (pi**3) * E))**0.25, 0)
    #取大值
    print(f"由挫曲負荷估算導螺桿桿徑: {dr_p}mm, 由導螺桿臨界轉速估算導螺桿桿徑: {dr_n}mm")
    dr_F =  max(dr_n, dr_p)
    #由DN估算導螺桿桿徑
    dr_DN = round(150000 / N, 0)
    print(f"直徑下限: {dr_F}mm, 直徑上限: {dr_DN}mm")
    print(f"{dr_F}mm < 螺桿直徑 < {dr_DN}mm")

    d_list = [12, 14, 15, 16, 20, 25, 28, 32, 36, 40, 45, 50, 55, 63, 70, 80, 100]
    suitable_dr = []
    cunt = 0
    found_any = False
    for diameter in range(len(d_list)):
        # 同時符合強度要求 (dr_F) 且在轉速限制內 (dr_DN)
        
        if d_list[diameter] >= dr_F and d_list[diameter] <= dr_DN:
            suitable_dr.append(d_list[diameter])
            cunt = diameter
            found_any = True
    if found_any and cunt+1 < len(d_list):
        suitable_dr.append(d_list[cunt+1])
    print(suitable_dr)

    return dr_F, dr_DN, suitable_dr
dr_F, dr_DN, suitable_dr = Diameter_calculation()
print("="*100)
print(f"導程: {guide}")
print("="*100)
#def Load_calculation():
    

由挫曲負荷估算導螺桿桿徑: 15.0mm, 由導螺桿臨界轉速估算導螺桿桿徑: 8.0mm
直徑下限: 15.0mm, 直徑上限: 38.0mm
15.0mm < 螺桿直徑 < 38.0mm
[15, 16, 20, 25, 28, 32, 36, 40]
導程: 12.0


In [ ]:
#動負荷計算
if gravity_axis_YN:
        p = (load + cutting_force)
else:
    cof = 0.008 #摩擦力係數
    ff = load * cof 
    p = (cutting_force + ff)

c = round(p / 3 / preload_rate, 0)
print(f"動負荷: {c} kfg")   

動負荷: 7453.0 kfg


In [5]:
#馬達扭矩計算
def Motor_torque_calculation(guide, load, cutting_force):
    w2 = load
    hsp = guide
    cof = 0.008
    me = 0.9
    Tt1 = (1 + cof) * hsp * w2 / (2 * pi * me) #kgf*mm
    Tt2 = abs((1 - cof) * hsp * w2 / (2 * pi * me)) #kgf*mm
    Tt = round(max(Tt1, Tt2) * 9.8 * 1e-3, 2) #N*m
    
    fc = cutting_force
    Tc = round((fc * hsp * me) / (2*pi)* 9.8 * 1e-3, 2) #N*m

    Trf = Tc + Tt
    print(f"移動件所引起的摩擦扭矩: {Tt} N．mm, 軸向力引起的扭矩:{Tc} N．mm")
    print(f"外加負荷引起之扭矩: {Trf} N．mm")
    return Trf

#馬達慣量計算
def Motor_inertia_calculation(length, suitable_dr, load, guide):
    proportion = 0.0078
    L = length
    g = 980
    Js = pi * proportion * (L* 0.1 )* ((suitable_dr[-1]*0.1)**4) / (32* g ) # kgf*cm*s**2
    W = load
    hsp = guide*0.1
    Jt = W / g * (hsp / 2 / pi)**2 #增加減速比考慮
    JL = round(Js + Jt, 4)
    print(f"負載慣量: {JL} kgf．cm．s2")
    return JL
Trf = Motor_torque_calculation(guide, load, cutting_force)
print("=" *100)
JL = Motor_inertia_calculation(length, suitable_dr, load, guide)


移動件所引起的摩擦扭矩: 16.25 N．mm, 軸向力引起的扭矩:5.78 N．mm
外加負荷引起之扭矩: 22.03 N．mm
負載慣量: 0.0473 kgf．cm．s2


In [ ]:
from img2table.document import PDF
from img2table.ocr import PaddleOCR
import warnings
import os

# 1. 忽略所有 UserWarning (包含 ccache 的提示)
warnings.filterwarnings("ignore", category = UserWarning)


paddle_ocr = PaddleOCR(lang="ch", kw={"use_angle_cls": True})

pdf = PDF(src = R"C:\Users\e11338\Downloads\上銀滾珠螺桿 預壓0.3.pdf")

extracted_tables = pdf.extract_tables(
    ocr = paddle_ocr, # 或 tesseract
    implicit_rows = True,         # [重要] 即使沒橫線也能根據文字對齊判斷行
    implicit_columns = True,      # [重要] 自動判斷縱向對齊，防止數據跑位
    borderless_tables = True,     # [重要] 偵測那些線條較細或不完整的邊框
    min_confidence = 30           # 調低信心門檻，先抓到資料，後續再用 Python 清洗
)

# for page, tables in extracted_tables.items():
#     for table in tables:
#         # 將結果轉為 pandas DataFrame 方便處理
#         df = table.df
#         print(f"Page {page} Table:")
#         display(df)

first_df = list(extracted_tables.values())[0][-1].df

# 印出結果
display(first_df)

In [3]:
import pandas as pd
import camelot

# 1. pages='all' 會抓取 PDF 內所有偵測到的表格 
tables = camelot.read_pdf(R"C:\Users\e11338\Downloads\上銀滾珠螺桿 預壓0.3.pdf", 
                          flavor='lattice', 
                          process_background=True,
                          line_scale=40,
                          pages='all')

print(f"總共偵測到 {len(tables)} 個表格區塊")

# 2. 合併所有表格 
# 注意：上銀型錄每頁的欄位結構可能略有不同（例如 FSV 與 FSI 型），
# 建議先檢查欄位數量是否一致再合併。
all_dfs = [t.df for t in tables]
full_df = pd.concat(all_dfs, ignore_index=True)

# 3. 顯示前 50 行檢查 
display(full_df.head(50))

總共偵測到 45 個表格區塊


,0,1,2,3,4,5,6,7,8,9,...,15,16,17,18,19,20,21,22,23,24
0,S \nT,,,,,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Z,ØX,,,,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,,,,,,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,ØDg6,ØD -0.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,型號,規格,,珠徑,PCD,根徑,珠卷數,剛性\nKgf /μm\nK,動負荷\nC ( kgf ),靜負荷\nCo ( kgf ),...,迴流管,,法蘭孔,,,接觸\n面長,NaN,NaN,NaN,NaN
5,,公稱\n外徑,導程,,,,,,,,...,W,H,X\nY,,Z,S,NaN,NaN,NaN,NaN
6,16-4B2,16,4,2.381,16.25,13.792,2.5x2,26,802,1722,...,23,21,5.5,9.5,5.5,12,NaN,NaN,NaN,NaN
7,16-5B1,,5,3.175,16.6,13.324,2.5x1,16,763\n1400,,...,27,22,5.5,9.5,5.5,12,NaN,NaN,NaN,NaN
8,16-5B2,,,,16.6,13.324,2.5x2,33,1385\n2799,,...,27,22,5.5,9.5,5.5,12,NaN,NaN,NaN,NaN
9,16-5C1,,,,16.6,13.324,3.5x1,22,1013,1946,...,27,22,5.5,9.5,5.5,12,NaN,NaN,NaN,NaN


In [79]:
import pandas as pd
#display(tables[44].df)
page = [2, 6, 10, 13, 16, 19, 23, 27, 31, 36, 40, 42, 44]
idx_list = []
for i in range(len(tables)):
    if len(tables[i].df) > 10:
        idx_list.append(i)

print(idx_list, len(idx_list))
for i in idx_list:
    print(len(tables[i].df))
    display(tables[i].df.head())

#len(page)
#tables[2].to_excel(R"C:\Users\e11338\Desktop\Feed System GAI\test_table.xlsx", sheet_name='Sheet1', index=True)

[2, 6, 10, 13, 16, 19, 23, 27, 31, 36, 40, 41, 42, 43, 44] 15
41


,0,1,2,3,4,5,6,7,8,9,...,11,12,13,14,15,16,17,18,19,20
0,型號,規格,,珠徑,PCD,根徑,珠卷數,剛性\nKgf /μm\nK,動負荷\nC ( kgf ),靜負荷\nCo ( kgf ),...,,法蘭,,,迴流管,,法蘭孔,,,接觸\n面長
1,,公稱\n外徑,導程,,,,,,,,...,L,F,T\nBCD-E,,W,H,X\nY,,Z,S
2,16-4B2,16,4,2.381,16.25,13.792,2.5x2,26,802,1722,...,48,52,10\n40,,23,21,5.5,9.5,5.5,12
3,16-5B1,,5,3.175,16.6,13.324,2.5x1,16,763\n1400,,...,45,54,12\n41,,27,22,5.5,9.5,5.5,12
4,16-5B2,,,,16.6,13.324,2.5x2,33,1385\n2799,,...,60,54,12\n41,,27,22,5.5,9.5,5.5,12


41


,0,1,2,3,4,5,6,7,8,9,...,11,12,13,14,15,16,17,18,19,20
0,型號,規格,,珠徑,PCD,根徑,珠卷數,剛性\nKgf /μm\nK,動負荷\nC ( kgf ),靜負荷\nCo ( kgf ),...,,法蘭,,,迴流管,,法蘭孔,,,接觸\n面長
1,,公稱\n外徑,導程,,,,,,,,...,L,F\nT,,BCD-E,W\nH,,X\nY,,Z,S
2,36-10B2,36,10,6.350,37.4,30.91,2.5x2,68,5105,12669,...,102,104,18,82\n49,,40,11,17.5,11,15
3,40-5B2,40,5,3.175,40.6,37.324,2.5x2\n66,,2071,7134,...,65,92,16,72\n46,,34,9,14,8.5,15
4,40-6B2,,6,3.969,40.8,36.744,2.5x2\n69,,2817,8855,...,72,94,16,76\n47,,36,9,14,8.5,15


21


...

,0,1,2,3,4,5,6,7,8,9,...,15,16,17,18,19,20,21,22,23,24
0,型號,規格,,節圓\n直徑,,根徑 珠徑,卷數,剛性 K\n(kgf/μm),動負荷\nC(kgf),靜負荷\nCo(kgf),...,,,,,,油孔,,,雙牙,不出牙
1,,公稱\n外徑,導程,,,,,,,,...,Form \nB (L8),Form \nC (L9),L7,D4,D5,M,,L10 L11,,
2,14-10K3,14,10,14.6,10.724,3.175,3,24,920,1790,...,40,44,,38\n45,,,6,5,,
3,15-10K3,15,10,16,12.869,3,3,26,930,1970,...,,,,,,,,,,
4,15-16K2,,16,,,,2,16,610,1230,...,,,,,,,,,,


21


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16
0,D6\nForm A\nTYPE 1\nTYPE 2\nL2\nM\n22.5°\n30°\...,,,,,,,,,,,,,,,,
1,,,,,,,,,,,,,,,,,
2,,,,,,L7\nL1,,,,,,,,D6,,,
3,,,,,,L11,,,,,,,,,,,
4,,,,,,,,,,,,,,,,,


61


,0,1,2,3,4,5,6,7,8,9,...,15,16,17,18,19,20,21,22,23,24
0,型號,規格,,節圓\n直徑,根徑,珠徑,卷數,剛性 K\n(kgf/μm),動負荷\nC(kgf),靜負荷\nCo(kgf),...,,,,,,油孔,,,雙牙,不出牙
1,,公稱\n外徑,導程,,,,,,,,...,Form \nB (L8),Form \nC (L9),L7,D4,D5,M,L10,L11,,
2,40-5K5,40,5,40.6,37.324,3.175,5,85,2470,9490,...,70,81.5,14,78,9,,,7,,
3,40-6K5,,6,40.8,36.744,3.969,5,95,3370,11780,...,,,,,,,,,,
4,40-8K5,,8,41,36.132,4.763,5,101,4360,14200,...,,,,,,,,,,●


In [187]:
import numpy as np
page = [2, 6, 10, 13, 16, 19, 23, 27, 31, 36, 40, 42, 44]
type = ["FSV", "FSI", "RSI", "FSC"]
pages = [6, 3, 2, 2 ]

def Data_cleanong(tables):
    cols_name = ["型號", "公稱 外徑", "導程", "珠徑", "PCD", "根徑", "珠卷數", "剛性 kfg/umk", "動負荷 C (kfg)", "靜負荷 Co (kfg)"]
    df = tables.df[2:]
    df = df.iloc[:, :10]
    df.columns = cols_name

        
    def split_all_merged_cells(df):
        # 建立一個副本，避免更動原始數據
        new_df = df.copy()
        
        # 遍歷每一列
        for index, row in new_df.iterrows():
            # 遍歷每一欄 (除了最後一欄，因為最後一欄沒人可以推擠)
            rows_count = len(new_df)
            cols_count = len(new_df.columns)
        for r in range(rows_count):
            for c in range(cols_count - 1): # 到倒數第二欄為止
                cell_val = new_df.iat[r, c]
                
                if '\n' in cell_val:
                    parts = cell_val.split('\n')
                    # 前半段留在原地
                    new_df.iat[r, c] = parts[0].strip()
                    # 後半段推擠到右邊那格
                    # 注意：如果右邊原本有值且非空白，這會覆蓋它。
                    # 但在上銀型錄中，被合併的右邊通常是空的。
                    new_df.iat[r, c + 1] = parts[1].strip()
                    
        return new_df

    df = split_all_merged_cells(df)
    df = df.replace(r'^\s*$', np.nan, regex=True)
    df = df.ffill()
    col = ["公稱 外徑", "導程", "動負荷 C (kfg)"]
    for col in col:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    return df

df = Data_cleanong(tables[36])
df
#df.to_excel(R"C:\Users\e11338\Desktop\Feed System GAI\test_table.xlsx", sheet_name='Sheet1', index=True)

,型號,公稱 外徑,導程,珠徑,PCD,根徑,珠卷數,剛性 kfg/umk,動負荷 C (kfg),靜負荷 Co (kfg)
2,16-2T4,16,2,1.500,16.2,14.652,4,15,178,395
3,16-5T3,16,5,3.175,16.6,13.324,3,11,731,1331
4,16-5T4,16,5,3.175,16.6,13.324,4,12,936,1775
5,20-5T3,20,5,3.175,20.6,17.324,3,20,852,1767
6,20-5T4,20,5,3.175,20.6,17.324,4,27,1091,2356
7,20-6T3,20,6,3.969,20.8,16.744,3,20,1091,2081
8,20-6T4,20,6,3.969,20.8,16.744,4,27,1398,2774
9,25-5T3,25,5,3.175,25.6,22.324,3,28,977,2314
10,25-5T4,25,5,3.175,25.6,22.324,4,37,1252,3085
11,25-6T3,25,6,3.969,25.8,21.744,3,28,1272,2762


In [202]:
page = [2, 6, 10, 13, 16, 19, 23, 27, 31, 36, 40]
type = ["FSV", "FSI", "RSI"]
group = [6, 3, 2]
conact = []
file_path = Rf"C:\Users\e11338\Desktop\Feed System GAI\HIWIN_Specs.xlsx"

with pd.ExcelWriter(file_path, engine='xlsxwriter') as writer:
    for idx, i in enumerate(group):
        conact = []
        for j in range(i):
            df = Data_cleanong(tables[page[j]])
            conact.append(df)
        # print(conact)
        # for i in range(10):
        #     print()
        df_conat = pd.concat(conact)
        globals()[f"{type[idx]}"] = df_conat
        globals()[f"{type[idx]}"].to_excel(writer, sheet_name = f"{type[idx]}", index=False)
        page = page[j+1:]

